# Model Explainability

Goals:
- Global feature importance (mean |SHAP|)
- SHAP beeswarm summary plot
- Local explanation for a single customer
- Demonstrate high-risk vs low-risk explanations

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## Load trained model and test data

In [ ]:
from src.models.registry import load_artifact

model        = load_artifact('xgb_calibrated')
scaler       = load_artifact('scaler')
feature_names = load_artifact('feature_names')

print(f'Loaded model. Features: {len(feature_names)}')

In [ ]:
from src.ingestion.load_data import load_raw
from src.processing.preprocess import clean_raw, apply_scaler
from src.features.feature_store import add_derived_features
from src.processing.split import split_dataset, get_X_y

df_raw = load_raw()
df     = add_derived_features(clean_raw(df_raw))
_, _, df_test = split_dataset(df)
X_test_raw, y_test = get_X_y(df_test)
X_test = apply_scaler(X_test_raw, scaler)
print(f'Test set: {X_test.shape}')

## Global feature importance

In [ ]:
from src.explainability.shap_analysis import global_feature_importance

importance = global_feature_importance(model, X_test, feature_names)
print(importance.head(15).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
top10 = importance.head(10)
ax.barh(top10['feature'][::-1], top10['mean_abs_shap'][::-1], color='steelblue')
ax.set_xlabel('Mean |SHAP Value|')
ax.set_title('Global Feature Importance (Top 10)')
plt.tight_layout()
plt.savefig('../reports/figures/shap_global_importance.png', dpi=150)
plt.show()

## SHAP beeswarm summary

In [ ]:
from src.explainability.shap_analysis import plot_shap_summary
from pathlib import Path

plot_shap_summary(model, X_test, feature_names,
                  save_path=Path('../reports/figures/shap_beeswarm.png'))

## Local explanation: single customer

In [ ]:
from src.explainability.shap_analysis import explain_single

# Pick a high-risk customer
proba = model.predict_proba(X_test)[:, 1]
high_risk_idx = proba.argmax()
low_risk_idx  = proba.argmin()

for label, idx in [('HIGH RISK', high_risk_idx), ('LOW RISK', low_risk_idx)]:
    row = X_test.iloc[[idx]]
    drivers = explain_single(model, row, feature_names, top_n=5)
    p = proba[idx]
    print(f'\n=== {label} (score={round(p*100)}, prob={p:.2f}) ===')
    for d in drivers:
        sign = '+' if d['shap_value'] > 0 else '-'
        print(f"  {sign} {d['feature']}: {d['direction']} (SHAP={d['shap_value']:+.3f}, value={d['feature_value']:.2f})")